<div class="alert alert-warning">

# PS 11 — Working Memory Models

In this problem set you will implement and compare two computational accounts of **visual working memory** (VWM): 
1. A behavioral mixture model that captures the capacity/resource debate
2. A neural network model that gives a mechanistic account of how the brain might maintain information over a delay.

## The paradigm

A participant briefly sees $N$ coloured items, waits through a blank delay, and is then probed on one item — reporting its colour as precisely as possible on a colour wheel. The key question is: **what limits performance as $N$ grows?**

Two competing theories:

- **Slot model** (Cowan, 2001): a fixed number $K$ of items are stored at full precision; items beyond capacity are lost entirely.
- **Resource model** (Bays & Husain, 2008): a continuous resource is divided equally across all $N$ items; every item is stored, but with precision that decreases with $N$.

We will:
1. Simulate both models in a change-detection task and compare their predictions
2. Fit a mixture model to continuous-recall data and distinguish the two accounts
3. Explore a ring-attractor neural network that provides a mechanistic basis for working memory maintenance

**Default parameters (unless stated otherwise):** capacity $K = 3$, maximum precision $\kappa_{max} = 8$, resource $W = 3$.

</div>

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats   import norm, vonmises
from scipy.special import i0

from numpy.testing import assert_allclose

---

# 1. Change Detection: Slot vs. Resource Models

![](WM_models.png)

In a change-detection trial the participant sees and attempts to memorize $N$ items. After a delay period, they judge whether a single probed item changed. We model accuracy (proportion correct on change trials) under each theory.

**Slot model** (Cowan, 2001): the probe item is in memory with probability $\min(1,\, K/N)$. If in memory, the change is detected perfectly; otherwise the participant guesses (50% correct):

$$P_\text{slot}(\text{correct}) = \frac{\min(K,N)}{N} + \frac{\max(N-K,\,0)}{N}\cdot 0.5$$

**Resource model** (Bays & Husain, 2008): precision $\kappa(N) = W/N$ decreases with load. The probability of detecting a change of magnitude $\Delta$ is a signal-detection problem with $d' = \kappa(N)\cdot\Delta$:

$$P_\text{resource}(\text{correct}) = \Phi\!\left(\frac{W}{N}\right)$$

where $\Phi$ is the standard normal CDF (we absorb $\Delta$ into $W$). In python, you can compute this function as norm.pdf. 

<div class="alert alert-success">

## Exercise 1A

**A.** Implement `slot_model(N, K=3)` and `resource_model(N, W=3.0)` returning $P(\text{correct})$ for a single set size $N$.

</div>

In [ ]:
K_DEFAULT = 3
W_DEFAULT = 3.0


def slot_model(N, K=K_DEFAULT):
    #(TODO: replace pass with your implementation)
    pass


def resource_model(N, W=W_DEFAULT):
    #(TODO: replace pass with your implementation)
    pass


# slot_model: default K = 3
assert_allclose(slot_model(1), 1.0)
assert_allclose(slot_model(3), 1.0)
assert_allclose(slot_model(6), 0.75)
# optional: explicit K (capacity edge)
assert_allclose(slot_model(4, K=4), 1.0)
assert_allclose(slot_model(8, K=4), 0.75)
# resource_model: default W = 3.0
assert_allclose(resource_model(1), norm.cdf(3.0))
assert_allclose(resource_model(3), norm.cdf(1.0))
assert_allclose(resource_model(6), norm.cdf(0.5))
print("Success!")

<div class="alert alert-success">

## Exercise 1B
**B.** Plot both curves for $N \in \{1,\ldots,8\}$ on the same axes. Label axes and add a legend.
</div>

In [ ]:
# Exercise 1B: plot
set_sizes  = np.arange(1, 9)
p_slot     = 'YOUR_CODE_HERE' #Hint: leverage slot_model
p_resource = 'YOUR_CODE_HERE' #Hint: leverage resource_model

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(set_sizes, p_slot,     'o-', color='steelblue', label='Slot model')
ax.plot(set_sizes, p_resource, 's-', color='tomato',    label='Resource model')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Chance')
ax.set_xlabel('Set size $N$')
ax.set_ylabel('P(correct)')
ax.set_title('Change detection: slot vs. resource model')
ax.set_xticks(set_sizes)
ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 1C

**C.** At which set size do the two models diverge most? Why does the divergence occur there?

</div>

**1C. Your Answer:** 

---

# 2. Continuous Recall and the Mixture Model
![](task_color.png)

Change detection is a blunt measure. A richer paradigm uses **continuous recall**: participants report the exact colour of the probed item on a colour wheel, and we analyse the distribution of angular errors $\theta \in [-\pi, \pi)$.


The **Zhang & Luck (2008) mixture model** decomposes recall errors as:

$$p(\theta) = (1 - g)\,\mathcal{VM}(\theta;\,0,\,\kappa) + g\cdot\frac{1}{2\pi}$$

where the parameters are:

- $g \in [0,1]$ — probability the item was **not** stored (guess rate)
- $\kappa > 0$ — precision of stored memories (inverse variance of the von Mises distribution $\mathcal{VM}$)
- $\mathcal{VM}(\theta;\,0,\kappa)$ — von Mises pdf centred at 0 (the circular analogue of a Gaussian distribution).

The two theories predict different patterns as $N$ increases:

- **Slot model**: $g$ increases above $N = K$ (items start dropping out), $\kappa$ stays constant.
- **Resource model**: $g \approx 0$ for all $N$; $\kappa$ decreases monotonically.

<div class="alert alert-success">

## Exercise 2A

**A.** Implement `simulate_recall(N, n_trials=500, kappa_max=8, K=3)` that generates recall errors under the **slot model**: each trial, the probe item is in memory with probability $\min(1, K/N)$ (draw from $\mathcal{VM}(0,\kappa_{max})$), or is a uniform guess otherwise. Return an array of errors in $[-\pi,\pi)$.

*Hint:* use `vonmises.rvs(kappa)` and `np.random.uniform(-np.pi, np.pi)`.

`fit_mixture` is provided — it finds the maximum-likelihood $(g, \kappa)$ using numerical optimisation.

</div>

In [ ]:
# Provided: von Mises pdf and mixture-model fitter
from scipy.optimize import minimize

def vm_pdf(theta, kappa):
    return np.exp(kappa * np.cos(theta)) / (2 * np.pi * i0(kappa))


def fit_mixture(errors):
    """
    Maximum-likelihood fit of the von Mises mixture model.

    Minimises the negative log-likelihood
        -sum log[(1-g)*VM(theta; 0, kappa) + g/(2*pi)]
    over g in [0, 1] and kappa in [0.1, 30] using L-BFGS-B.

    Returns
    -------
    g_hat, kappa_hat : floats
    """
    def neg_ll(params):
        g, kappa = params
        pdf = (1 - g) * vm_pdf(errors, kappa) + g / (2 * np.pi)
        return -np.sum(np.log(pdf + 1e-300))

    result = minimize(neg_ll,
                      x0=[0.2, 5.0],
                      bounds=[(1e-6, 1 - 1e-6), (0.1, 30)],
                      method='L-BFGS-B')
    return float(result.x[0]), float(result.x[1])


KAPPA_MAX = 8
K_CAP     = 3


def simulate_recall(N, n_trials=500, kappa_max=KAPPA_MAX, K=K_CAP):
    errors = []
    for _ in range(n_trials):
        #(TODO: replace pass with your implementation)
        pass
    return np.array(errors)


# Sanity check
np.random.seed(42)
n = 50
errs = simulate_recall(N=4, n_trials=n, kappa_max=KAPPA_MAX, K=K_CAP)
assert errs.shape == (n,)
# scipy draws lie in [-pi, pi]; uniform uses [-pi, pi)
assert np.all(errs >= -np.pi) and np.all(errs <= np.pi)

e_test = simulate_recall(N=2, n_trials=300)
g_t, k_t = fit_mixture(e_test)
print(f"N=2 check: g={g_t:.2f}, kappa={k_t:.1f}   (expect g~0, kappa~{KAPPA_MAX})")

<div class="alert alert-success">

## Exercise 2B


**B.** Simulate 500 errors for $N \in \{1, 3, 6\}$ and plot a histogram of the angular errors for each set size (1 row $\times$ 3 subplots, shared y-axis). Add a horizontal dashed line at the uniform density $\frac{1}{2\pi}$. Describe how the distribution changes as $N$ increases.

</div>

In [ ]:
# Exercise 2B: visualise recall error distributions for N = 1, 3, 6
np.random.seed(0)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)

for ax, N_plot in zip(axes, [1, 3, 6]):
    errors_plot = 'YOUR_CODE_HERE'
    # (TODO: plot a histogram with ax.hist...)

    ax.axhline(1 / (2 * np.pi), color='tomato', linestyle='--',
               linewidth=1.5, label='Uniform')
    ax.set_xlabel('Recall error (rad)')
    ax.set_title(f'N = {N_plot}')
    ax.set_xlim(-np.pi, np.pi)
    if N_plot == 1:
        ax.legend(fontsize=8)

axes[0].set_ylabel('Density')
fig.suptitle('Recall error distributions (slot model, K=3)', y=1.02)
plt.tight_layout()
plt.show()

**2B. Your Answer:** 

<div class="alert alert-success">

## Exercise 2C

**C.** Simulate data for $N \in \{1, 2, 3, 4, 5, 6\}$ and fit the mixture model to each. Plot $\hat{g}$ and $\hat{\kappa}$ versus $N$ in two side-by-side subplots.

</div>

In [ ]:
# Exercise 2C: fit mixture model for N = 1..6
np.random.seed(0)
set_sizes    = np.arange(1, 7)
fitted_g     = []
fitted_kappa = []

for N in set_sizes:
    errors       = 'YOUR_CODE_HERE' #(TODO)
    g_hat, k_hat = 'YOUR_CODE_HERE' #(TODO)
    fitted_g.append(g_hat)
    fitted_kappa.append(k_hat)
    print(f"N={N}: g={g_hat:.2f}, kappa={k_hat:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# (TODO: plot fitted_g with axes[0].plot...)
axes[0].axvline(K_CAP, color='gray', linestyle='--', label=f'K={K_CAP}')
axes[0].set_xlabel('Set size $N$')
axes[0].set_ylabel('Fitted guess rate')
axes[0].set_title('Guess rate vs. set size')
axes[0].legend()

# (TODO: plot fitted_kappa with ax.axes[1].plot...)
axes[1].set_xlabel('Set size $N$')
axes[1].set_ylabel('Fitted precision')
axes[1].set_title('Precision vs. set size')
axes[1].set_ylim(0, 10)
axes[1].hlines(8, 1, 6, color='gray', linestyle='--')

plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 2D

**D.** In 1–2 sentences: what pattern would you expect if data came from the **resource** model instead?

</div>

**2D. Your Answer:** 

---

# 3. Ring Attractor Network

How might the brain actually *hold* information across a delay? One influential proposal is the **ring attractor**: a population of neurons, each tuned to a different feature value (e.g. colour or orientation), connected via **Mexican-hat** recurrent weights — nearby neurons excite each other, distant neurons inhibit. A brief input creates a localised bump of activity that persists long after the stimulus is gone, providing a neural substrate for working memory.

## The network

$N = 100$ neurons are arranged on a ring with preferred orientations $\theta_i \in [0, 2\pi)$. Recurrent weights follow a Mexican-hat profile:

$$W_{ij} = J_\text{exc}\,\exp\!\left(-\frac{d(\theta_i,\theta_j)^2}{2\sigma^2}\right) - J_\text{inh}$$

where $d(\cdot,\cdot)$ is the angular distance, $J_\text{exc}$ is the excitatory peak, $J_\text{inh}$ is global inhibition, and $\sigma$ controls the width of excitatory connections.

Activity evolves as:

$$r(t+1) = \operatorname{clip}\!\bigl[(1-\alpha)\,r(t) + \alpha\,\bigl(W r(t) + I(t) + \eta(t)\bigr),\;0,\;1\bigr]$$

where $\alpha$ is the update rate and $\eta \sim \mathcal{N}(0,\sigma_\eta^2)$ is noise.

The network (`make_weights`, `run_attractor`, `plot_activity`) is provided. Your task is to explore how its behaviour depends on parameters.


In [ ]:
# ── PROVIDED: Ring Attractor Network ────────────────────────────────────────
N     = 100
THETA = np.linspace(0, 2 * np.pi, N, endpoint=False)


def _angular_dist(a, b):
    d = a - b
    return (d + np.pi) % (2 * np.pi) - np.pi


def make_weights(J_exc=1.5, J_inh=0.5, sigma=0.5):
    """
    Build the Mexican-hat weight matrix.
    J_exc  - peak excitatory strength
    J_inh  - uniform global inhibitory strength
    sigma  - width of excitatory connections (radians)
    """
    d = _angular_dist(THETA[:, None], THETA[None, :])
    return J_exc * np.exp(-d ** 2 / (2 * sigma ** 2)) - J_inh


def run_attractor(W, cues, T=200, alpha=0.02, noise_std=0.0, seed=0):
    """
    Simulate the ring attractor.

    W         - (N, N) weight matrix from make_weights()
    cues      - list of dicts, each with:
                  pos      - cue centre (radians)
                  strength - cue amplitude
                  on, off  - time steps the cue is active [on, off)
    T         - total time steps
    alpha     - update rate (controls how fast activity changes)
    noise_std - std of Gaussian noise added each step
    seed      - random seed

    Returns (T, N) array of firing rates.
    """
    rng = np.random.default_rng(seed)
    r   = np.zeros(N)
    history = np.zeros((T, N))

    for t in range(T):
        I = np.zeros(N)
        for cue in cues:
            if cue['on'] <= t < cue['off']:
                d  = _angular_dist(THETA, cue['pos'])
                I += cue['strength'] * np.exp(-d ** 2 / (2 * 0.3 ** 2))

        noise = noise_std * rng.standard_normal(N)
        r     = np.clip((1 - alpha) * r + alpha * (W @ r + I) + noise, 0, 1)
        history[t] = r

    return history


def plot_activity(history, title='', cue_off=None, ax=None):
    """
    Plot network activity as a heatmap (time x preferred orientation).
    cue_off - if given, draw a vertical dashed line at this time step.
    ax      - existing Axes to plot into (creates a new figure if None).
    """
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.imshow(history.T, aspect='auto', origin='lower', cmap='hot',
                   extent=[0, history.shape[0], 0, 360], vmin=0, vmax=1)
    if cue_off is not None:
        ax.axvline(cue_off, color='cyan', linestyle='--',
                   linewidth=1.5, label='Cue off')
        ax.legend(loc='upper right', fontsize=8)
    plt.colorbar(im, ax=ax, label='Firing rate')
    ax.set_xlabel('Time step')
    ax.set_ylabel('Preferred orientation (deg)')
    ax.set_title(title)
    if own_fig:
        plt.tight_layout()
        plt.show()


W_default = make_weights()
print("Network ready. Default: J_exc=1.5, J_inh=0.5, sigma=0.5")

In [ ]:
# Visualise the Mexican-hat weight profile
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: 1-D profile (weights from the neuron at 0 deg to every other neuron)
ref = 0
d_deg = np.degrees(_angular_dist(THETA, THETA[ref]))   # angular distance in deg
order = np.argsort(d_deg)
axes[0].plot(d_deg[order], W_default[ref, order], color='steelblue', linewidth=2)
axes[0].axhline(0, color='k', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Angular distance from reference neuron (deg)')
axes[0].set_ylabel('Synaptic weight')
axes[0].set_title('Mexican-hat weight profile')
axes[0].set_xticks([-180, -90, 0, 90, 180])

# ── Right: 2-D weight matrix as a heatmap
im = axes[1].imshow(W_default, aspect='auto', cmap='RdBu_r', origin='lower',
                    extent=[0, 360, 0, 360])
plt.colorbar(im, ax=axes[1], label='Synaptic weight')
axes[1].set_xlabel('Pre-synaptic neuron (deg)')
axes[1].set_ylabel('Post-synaptic neuron (deg)')
axes[1].set_title('Full weight matrix')

plt.tight_layout()
plt.show()

<div class="alert alert-success">

## Exercise 3A

**A.** Run the network with default parameters and a single cue at $\theta = \pi$ (180 deg), active for steps 0-30, $T=200$ total. Use `plot_activity` with `cue_off=30`. Describe what happens before, during, and after the cue.

</div>

In [ ]:
# Exercise 3A: single cue, no noise
cues_A = [{'pos': np.pi, 'strength': 3.0, 'on': 0, 'off': 30}]
# (TODO) use run_attractor to simulate the network activity

# (TODO) use plot_activity to visualise the activity

**3A. Your Answer:** 

<div class="alert alert-success">

## Exercise 3B 

Repeat with `noise_std = 0.1` and `noise_std = 0.2` (use `seed=0`). What happens to the bump? What aspect of working memory does this capture?

</div>

In [ ]:
# Exercise 3B: noise causes bump drift
noise_levels = [0.0, 0.1, 0.2]
seed        = 0
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, noise in zip(axes, noise_levels):
    # (TODO) use run_attractor to simulate the network activity

    # (TODO) use plot_activity to visualise the activity
    pass
plt.tight_layout()
plt.show()

**3B. Your Answer:** 

<div class="alert alert-success">

## Exercise 3C

Present **two simultaneous cues** at $\theta = \pi/2$ and $\theta = 3\pi/2$, both active for steps 0-30. What happens after the cues are removed? Then try **sequential** cues (first cue steps 0-30, second cue steps 50-80). Use `strength=15` for both cues in this exercise. Does the second cue create a persistent bump? Why or why not? 


</div>

In [ ]:
# Exercise 3C: two cues  (use strength=15 so Cue 2 can compete with the established bump)
CUE_STR = 15.0
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Simultaneous
cues_sim = [{'pos': np.pi / 2,     'strength': CUE_STR, 'on':  0, 'off': 30},
            {'pos': 3 * np.pi / 2, 'strength': CUE_STR, 'on':  0, 'off': 30}]

# (TODO) use run_attractor to simulate the network activity

# (TODO) use plot_activity to visualise the activity

# Sequential
cues_seq = [{'pos': np.pi / 2,     'strength': CUE_STR, 'on':  0, 'off': 30},
            {'pos': 3 * np.pi / 2, 'strength': CUE_STR, 'on': 50, 'off': 80}]
# (TODO) use run_attractor to simulate the network activity

# (TODO) use plot_activity to visualise the activity
axes[1].axvline(30, color='cyan',   linestyle='--', linewidth=1.5, label='Cue 1 off')
axes[1].axvline(50, color='yellow', linestyle='--', linewidth=1.5, label='Cue 2 on')
axes[1].axvline(80, color='orange', linestyle='--', linewidth=1.5, label='Cue 2 off')
axes[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

**3C. Your Answer:** 

<div class="alert alert-success">

## Exercise 3D
    
**D.** In the previous sequential case exercise, still using `strength=15`, vary the strength of the second cue by a factor of 1 to 2 times the first cue. What happens?

</div>

In [ ]:
# Exercise 3D: vary the strength of Cue 2 relative to Cue 1 (strength=15)
factors = [1.0, 1.5, 1.75, 2.0]
fig, axes = plt.subplots(1, len(factors), figsize=(16, 4))

for ax, factor in zip(axes, factors):
    s2 = 15.0 * factor
    cues = [{'pos': np.pi / 2,     'strength': 15.0, 'on':  0, 'off': 30},
            {'pos': 3 * np.pi / 2, 'strength': s2,   'on': 50, 'off': 80}]
    # (TODO) use run_attractor to simulate the network activity

    # (TODO) use plot_activity to visualise the activity
    
    ax.axvline(30, color='cyan',   linestyle='--', linewidth=1.2)
    ax.axvline(50, color='yellow', linestyle='--', linewidth=1.2)
    ax.axvline(80, color='orange', linestyle='--', linewidth=1.2)

plt.tight_layout()
plt.show()

**3D. Your Answer:** 

<div class="alert alert-success">

## Exercise 3E

**E.** Vary one parameter at a time and report what you observe:
- Reduce $J_\text{exc}$ from 1.5 to 0.8 (all else default)
- Increase $J_\text{exc}$ to 3.0 (all else default)
- Increase $J_\text{inh}$ from 0.5 to 1.5 (all else default)


</div>

In [ ]:
# Exercise 3E: vary parameters
single_cue = [{'pos': np.pi, 'strength': 3.0, 'on': 0, 'off': 30}]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

W_low  = make_weights(J_exc=0.8, J_inh=0.5, sigma=0.5)
# (TODO) use run_attractor to simulate the network activity

# (TODO) use plot_activity to visualise the activity

W_high = make_weights(J_exc=3.0, J_inh=0.5, sigma=0.5)
# (TODO) use run_attractor to simulate the network activity

# (TODO) use plot_activity to visualise the activity

W_inh  = make_weights(J_exc=1.5, J_inh=1.5, sigma=0.5)
# (TODO) use run_attractor to simulate the network activity

# (TODO) use plot_activity to visualise the activity

plt.tight_layout()
plt.show()

**3E. Your Answer:**


<div class="alert alert-success">

## Exercise 3F

**F.** In 2-3 sentences, what does this model suggest about the neural basis of working memory capacity limits?

</div>

**3F. Your Answer:** 

---

<div class="alert alert-warning">


When you are done, **restart the kernel and re-run the whole notebook** (`Kernel -> Restart & Run All`) to make sure everything runs without errors.

</div>